# 🚀 Messung der OCR-Qualität in Python

```{admonition} Hinweise zur Ausführung des Notebooks
:class: hinweis
Dieses Notebook kann auf unterschiedlichen Levels erarbeitet werden (siehe Abschnitt ["Technische Voraussetzungen"](../introduction/introduction_requirements)):

1. **Book-Only Mode:** Sie lesen das Notebook hier im "Jupyter Book", ohne den Code selbst auszuführen.
2. **Cloud Mode:** Klicken Sie **oben rechts in der Menüleiste** auf das Raketen-Symbol <span class="launch-colab-inline">🚀</span> und wählen Sie "Colab", um das Notebook auszuführen.
3. **Local Mode:** Klicken Sie **oben rechts in der Menüleiste** auf das Download-Symbol <span class="launch-ipynb-inline">↓</span> und wählen Sie ".ipynb", um das Notebook lokal auszuführen.
```

## Übersicht
In diesem Notebook wird die Qualität von OCR-Ergebnissen gemessen und bewertet. Ziel ist es, den OCR-Output eines historischen Zeitungsartikels (in Frakturschrift) mit einem manuell erstellten Referenztext (Ground Truth) zu vergleichen und die Güte der Texterkennung anhand standardisierter Metriken zu quantifizieren.

Dafür werden folgende Schritte durchgeführt:
1. Durchführen der OCR auf einem Beispielbild mit Tesseract (Fraktur-Modell)
2. Manuelle Erstellung eines Ground-Truth-Textes als Referenz
3. Berechnung von Precision, Recall und F1-Score zum Vergleich von OCR-Output und Ground Truth

## Installationen und Importe

In [ ]:
# 🚀 Install libraries
import sys
if 'google.colab' in sys.modules:
    !sudo apt install tesseract-ocr
    !sudo apt install tesseract-ocr-frk
    !sudo apt install poppler-utils
!pip install pytesseract pillow Levenshtein requests

In [ ]:
import pytesseract
from PIL import Image
from pathlib import Path

Bevor wir Daten herunterladen, definieren wir eine kleine Hilfsfunktion `download_file`. Sie lädt eine Datei plattformunabhängig – also auch unter Windows – aus dem Internet in einen Zielordner herunter und ersetzt damit das Kommando `wget`, das nicht auf allen Systemen (z.B. Windows) nativ verfügbar ist.

In [ ]:
# helper: download a single file (cross-platform replacement for `! wget -P`)
import requests

def download_file(url, target_dir):
    """Download the file at `url` into `target_dir`, keeping its original name."""
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / url.split("/")[-1]
    response = requests.get(url)
    response.raise_for_status()
    target_path.write_bytes(response.content)
    return target_path

In [ ]:
# 🚀 get the image to process
if not Path("grippe.jpeg").exists():
    download_file("https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/refs/heads/main/ocr/grippe.jpeg", ".")

## Durchführen von OCR

Wir werden denselben Beispielartikel zur Bewertung verwenden:
<!-- In this notebook we evaluate OCR quality. We'll use the same sample article for evaluation: -->

<img src='grippe.jpeg' width="600">

Zuerst führen wir die OCR durch: <!-- Let us OCR it first: -->

In [ ]:
ocr_output = pytesseract.image_to_string(Image.open('grippe.jpeg'), 
                                         lang='frk') # Wir verwenden das Fraktur-OCR-Modell

In [ ]:
print(ocr_output)

### Manuell die 'Ground Truth' zur Bewertung erstellen <!-- ### Manually create  the 'ground truth' to evaluate against -->

In [ ]:
ground_truth = """Die Grippe wütet weiter 
Zunahme der schweren Fälle in Berlin. 
Die Zahl der Grippefälle ist in den letzten 
beiden Tagen auch in Groß-Berlin noch 
erheblich gestiegen. Die Warenhäuser und son-
stigen großen Geschäfte, die Kriegs- und die pri-
vaten Betriebe klagen, daß übermäßig viele An-
gestellte sich haben krank melden müssen und auch 
bei der Post und bei der Straßenbahn ist der 
Prozentsatz der Grippekranken bedeutend ge-
stiegen."""

In [ ]:
print(ground_truth)

## Berechnung von Precision, Recall und F1-Score <!-- ### Calculate precision, recall, and F1-score -->

In [ ]:
# 🚀 create folder and get the auxiliary python script to run in Colab
aux_dir = Path("auxiliary")
if not aux_dir.exists():
    aux_dir.mkdir(parents=True)

In [21]:
# 🚀 get the auxiliary python script
if not (aux_dir / "measure_ocr_quality.py").exists():
    download_file("https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/main/ocr/auxiliary/measure_ocr_quality.py", aux_dir)

In [ ]:
from auxiliary.measure_ocr_quality import measure_ocr_quality

In [ ]:
precision, recall, f_score = measure_ocr_quality(ocr_output, ground_truth)

In [ ]:
print(f'Precision: {round(precision, 4)}\nRecall: {round(recall, 4)}\nF1-score: {round(f_score, 4)}')